# IrisAI — Iris Flower Classification Notebook
## Comprehensive End-to-End Data Science & Machine Learning Workflow

This notebook demonstrates dataset loading, cleaning, exploratory data analysis (EDA), visualization, train/test splitting, feature scaling, model selection, hyperparameter tuning, and saving model artifacts for deployment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

## 1. Load and Inspect Dataset

In [ ]:
dataset_path = Path('../dataset/iris.csv')
if dataset_path.exists():
    df = pd.read_csv(dataset_path)
else:
    iris = load_iris()
    df = pd.DataFrame(iris.data, columns=['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm'])
    df['Species'] = [iris.target_names[i] for i in iris.target]

df.head()

## 2. Data Cleaning & Column Normalization

In [ ]:
# Normalize column names
column_mapping = {}
for col in df.columns:
    c = str(col).strip().lower()
    if 'sepallength' in c: column_mapping[col] = 'sepal_length'
    elif 'sepalwidth' in c: column_mapping[col] = 'sepal_width'
    elif 'petallength' in c: column_mapping[col] = 'petal_length'
    elif 'petalwidth' in c: column_mapping[col] = 'petal_width'
    elif 'species' in c: column_mapping[col] = 'species'
    elif c in ['id', 'index']: column_mapping[col] = 'DROP'

df = df.rename(columns=column_mapping)
if 'DROP' in df.columns:
    df = df.drop(columns=['DROP'])

# Check missing values and duplicates
print('Missing Values:\n', df.isnull().sum())
print('Duplicates:', df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
df.describe().T

In [ ]:
sns.pairplot(df, hue='species', palette='muted')
plt.suptitle('Iris Species Pairwise Relationships', y=1.02)
plt.show()

## 4. Feature Engineering & Train/Test Split

In [ ]:
X = df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]
le = LabelEncoder()
y = le.fit_transform(df['species'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Model Training, Tuning & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'Gaussian Naive Bayes': GaussianNB(),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    print(f'{name} -> Test Accuracy: {acc * 100:.2f}%')